<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.1-sft-lora/practice/GCP_Capstone_10.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 10.1 — SFT with LoRA

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: Install, authenticate, and initialize the client

Run this first. It installs the SDKs, authenticates with Application Default Credentials (Colab), and creates the Vertex AI `genai` client every later cell depends on.

In [ ]:
%%bash
pip install -q google-genai google-cloud-aiplatform pandas

In [ ]:
# Authenticate with Application Default Credentials (Colab only) — never API keys
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab ADC')
except ImportError:
    print('Not on Colab — assuming ADC is already configured (gcloud auth application-default login)')

from google import genai
from google.genai import types
import json, time, os

PROJECT = 'documind-ai-YOUR-ID'   # <-- your GCP project id
LOCATION = 'us-central1'          # course region (use asia-south1 for India prod)
BUCKET = 'your-bucket'            # Cloud Storage bucket (no gs:// prefix)
USD_INR = 85

client = genai.Client(enterprise=True, project=PROJECT, location=LOCATION)
# Tuning (client.tunings.*) is regional-only, so the client above stays regional.
# Gemini 3.x generation is served ONLY from the global endpoint, so route any
# generate_content calls through a separate global client.
gen_client = genai.Client(enterprise=True, project=PROJECT, location='global')
print('SDK ready')

## Exercise 1: Create a JSONL training file

**Difficulty:** Easy

Write 10 document classification examples in proper Gemini JSONL format with systemInstruction, user/model alternation.

1. Define document/label pairs across the five DocuMind categories.
2. Wrap each in a record with `systemInstruction`, then a `contents` list.
3. Alternate roles: `user` first, `model` last, using `parts` with a `text` field.
4. Serialize one JSON object per line and write `train.jsonl`.

**Expected behaviour:** `train.jsonl` with 10 valid JSON lines, last entry role=model.

In [ ]:
# DocuMind document classification training data (extended to 10 examples)
training_examples = [
    {'doc': 'Invoice #INV-2024-0892\nBill To: Acme Corp\nDate: March 15, 2024\nItem: Cloud Services - Q1\nAmount: $45,000.00\nPayment: Net 30', 'label': 'INVOICE'},
    {'doc': 'Service Agreement\nBetween: TechCorp and ClientX\nEffective: January 1, 2024\nTerm: 12 months\nThis agreement outlines the terms and conditions of service delivery...', 'label': 'CONTRACT'},
    {'doc': 'Q1 2024 Financial Report\nExecutive Summary: Revenue increased 23% YoY...\nOperating margin improved to 18%.\nKey insights and forward-looking statements follow.', 'label': 'REPORT'},
    {'doc': 'Receipt #R-2024-5521\nDate: 2024-03-20\nStore: OfficeSupplies Inc.\nItems: Printer paper - $25.50, Ink cartridges - $89.00\nTotal: $114.50\nPayment: Credit Card', 'label': 'RECEIPT'},
    {'doc': 'From: john@client.com\nTo: support@documind.ai\nSubject: Question about API rate limits\n\nHi team, I am hitting rate limits on my production deployment...', 'label': 'CORRESPONDENCE'},
    {'doc': 'Invoice #INV-2024-1130\nBill To: Globex Ltd\nDate: April 2, 2024\nItem: Support Retainer\nAmount: $12,000.00\nPayment: Net 15', 'label': 'INVOICE'},
    {'doc': 'Master Services Agreement\nBetween: DataWorks and NorthStar Inc.\nEffective: February 10, 2024\nTerm: 36 months\nSection 1: Scope of Services...', 'label': 'CONTRACT'},
    {'doc': 'Annual Report 2023\nExecutive Summary: Net income rose 11% to $4.2M.\nHeadcount grew from 210 to 265.\nOutlook for FY2024 remains positive.', 'label': 'REPORT'},
    {'doc': 'Receipt #R-2024-6710\nDate: 2024-04-11\nStore: CloudMart\nItems: USB-C hub - $39.99, HDMI cable - $14.50\nTotal: $54.49\nPayment: Debit Card', 'label': 'RECEIPT'},
    {'doc': 'From: priya@vendor.in\nTo: procurement@documind.ai\nSubject: Re: Renewal quote\n\nHello, attaching the revised quote for next year as discussed on the call...', 'label': 'CORRESPONDENCE'},
]

# Build JSONL with proper Gemini format
system_prompt = 'You are DocuMind, a document classifier. Classify documents as: INVOICE, CONTRACT, REPORT, RECEIPT, or CORRESPONDENCE. Respond with the category name only.'

jsonl_lines = []
for ex in training_examples:
    record = {
        'systemInstruction': {'role': 'system', 'parts': [{'text': system_prompt}]},
        'contents': [
            {'role': 'user', 'parts': [{'text': f'Classify:\n\n{ex["doc"]}'}]},
            {'role': 'model', 'parts': [{'text': ex['label']}]}
        ]
    }
    jsonl_lines.append(json.dumps(record))

with open('train.jsonl', 'w') as f:
    f.write('\n'.join(jsonl_lines))

print(f'Created train.jsonl with {len(jsonl_lines)} examples')
print('\nFirst example:')
print(jsonl_lines[0][:200] + '...')

## Exercise 2: Validate JSONL format

**Difficulty:** Easy

Write validation: roles alternate, last entry is model, no 'assistant' role, uses 'text' not 'content'.

1. Parse each line as JSON and collect the `contents` roles.
2. Assert the last role is `model` and that no role is `assistant`.
3. Assert roles strictly alternate `user`/`model`.
4. Flag any part that uses a `content` key instead of `text`.

**Expected behaviour:** Validation catches assistant role, content field mistakes.

In [ ]:
# Validation: catch common JSONL mistakes before training
def validate_jsonl(path):
    errors = []
    with open(path) as f:
        for i, line in enumerate(f, 1):
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f'Line {i}: Invalid JSON - {e}')
                continue

            # Check contents exists
            if 'contents' not in rec:
                errors.append(f'Line {i}: Missing contents field')
                continue

            contents = rec['contents']
            roles = [c.get('role') for c in contents]

            # Check last entry is model
            if roles[-1] != 'model':
                errors.append(f'Line {i}: Last entry must be role=model, got {roles[-1]}')

            # Check no assistant role
            if 'assistant' in roles:
                errors.append(f'Line {i}: Use role=model, not role=assistant')

            # Check roles alternate
            for j, r in enumerate(roles):
                expected = 'user' if j % 2 == 0 else 'model'
                if r != expected:
                    errors.append(f'Line {i}: Position {j} should be {expected}, got {r}')
                    break

            # Check parts use text field (common mistake: content)
            for j, c in enumerate(contents):
                for p in c.get('parts', []):
                    if 'content' in p and 'text' not in p:
                        errors.append(f'Line {i}: Use "text" not "content" in parts')

    return errors

errors = validate_jsonl('train.jsonl')
if errors:
    for e in errors[:5]:
        print('ERROR:', e)
else:
    print('OK: JSONL format is valid')

## Exercise 3: Calculate training cost

**Difficulty:** Easy

200 examples averaging 500 tokens each, 3 epochs. Estimate training token count and cost.

1. Read `train.jsonl` and approximate tokens as characters / 4.
2. Multiply by the number of epochs to get training tokens.
3. Multiply by the per-token tuning rate to get USD cost.
4. Also project the 200-example × 500-token × 3-epoch scenario.

**Expected behaviour:** 300,000 training tokens, ~$1-3 depending on model rate.

In [ ]:
# Estimate training cost before launching
def estimate_training_cost(jsonl_path, epochs=3, per_token_rate=0.0000035):
    '''Rough estimate: 1 token ~ 4 characters for English.'''
    total_chars = 0
    n_examples = 0
    with open(jsonl_path) as f:
        for line in f:
            rec = json.loads(line)
            total_chars += len(json.dumps(rec))
            n_examples += 1

    approx_tokens = total_chars // 4
    training_tokens = approx_tokens * epochs
    cost = training_tokens * per_token_rate

    print(f'Examples: {n_examples}')
    print(f'Approx tokens per example: {approx_tokens // n_examples if n_examples else 0}')
    print(f'Total approx tokens: {approx_tokens:,}')
    print(f'Training tokens (x {epochs} epochs): {training_tokens:,}')
    print(f'Estimated training cost: ${cost:.4f} (about Rs.{cost * USD_INR:.2f})')
    print(f'(Inference cost: SAME as base model - LoRA adapters are free to serve)')

estimate_training_cost('train.jsonl', epochs=3)

# Projected scenario from the exercise: 200 examples x 500 tokens x 3 epochs
projected_tokens = 200 * 500 * 3
projected_cost = projected_tokens * 0.0000035
print('\n--- Projected 200-example scenario ---')
print(f'Training tokens: {projected_tokens:,}')
print(f'Estimated cost: ${projected_cost:.2f} (about Rs.{projected_cost * USD_INR:.2f})')

## Exercise 4: Launch a tuning job

**Difficulty:** Medium

Use client.tunings.tune() with gemini-3.6-flash, epochs=3. Poll until JOB_STATE_SUCCEEDED.

1. Upload `train.jsonl` to your GCS bucket.
2. Call `client.tunings.tune()` with the base model, training dataset, and epoch config.
3. Poll `client.tunings.get()` until the job reaches a terminal state.
4. Read the tuned model endpoint off the finished job.

**Expected behaviour:** Tuning job completes, endpoint URI returned.

In [ ]:
%%bash
# Upload the training file to GCS (edit BUCKET first). Safe to skip if you are only defining functions.
gsutil cp train.jsonl gs://your-bucket/train.jsonl || echo 'Set BUCKET and re-run to actually upload'

In [ ]:
# Template: actual launch requires a valid GCS bucket and project
def launch_tuning_job(base_model='gemini-3.6-flash',
                      train_uri='gs://bucket/train.jsonl',
                      val_uri=None, epochs=3,
                      display_name='documind-classifier-v1'):
    '''Launch a supervised fine-tuning job.'''
    config_kwargs = {
        'epoch_count': epochs,
        'tuned_model_display_name': display_name,
    }

    dataset_kwargs = {'gcs_uri': train_uri}

    tuning_job = client.tunings.tune(
        base_model=base_model,
        training_dataset=types.TuningDataset(**dataset_kwargs),
        config=types.CreateTuningJobConfig(**config_kwargs)
    )
    return tuning_job

# Poll pattern
def poll_until_complete(tuning_job):
    completed = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'}
    while str(tuning_job.state) not in completed:
        print(f'Status: {tuning_job.state}')
        tuning_job = client.tunings.get(name=tuning_job.name)
        time.sleep(30)
    return tuning_job

print('Functions defined. Uncomment to launch (needs real bucket + project):')
print('# job = launch_tuning_job(train_uri=f"gs://{BUCKET}/train.jsonl", epochs=3)')
print('# job = poll_until_complete(job)')
print('# tuned_endpoint = job.tuned_model.endpoint')
print('# print(f"Tuned endpoint: {tuned_endpoint}")')

## Exercise 5: Function calling JSONL

**Difficulty:** Medium

Write training examples with top-level tools array, functionCall in model parts, functionResponse.

1. Add a top-level `tools` array with `functionDeclarations`.
2. Put the user's request in a `user` turn.
3. Emit a `model` turn whose part is a `functionCall` with `name` + `args`.
4. (Optional) Follow with a `functionResponse` turn to close the loop.

**Expected behaviour:** JSONL with functionDeclarations, functionCall name+args, functionResponse.

In [ ]:
# Function calling training example (pattern: model generates a function call)
fc_example = {
    'system_instruction': {
        'role': 'system',
        'parts': [{'text': 'You are DocuMind. Use the provided tools to process documents.'}]
    },
    'contents': [
        {'role': 'user', 'parts': [{'text': 'I uploaded a new document. Please classify it.'}]},
        {'role': 'model', 'parts': [{
            'functionCall': {
                'name': 'classify_document',
                'args': {'document_id': 'current_doc', 'confidence_threshold': 0.85}
            }
        }]},
        {'role': 'user', 'parts': [{
            'functionResponse': {
                'name': 'classify_document',
                'response': {'category': 'INVOICE', 'confidence': 0.97}
            }
        }]},
        {'role': 'model', 'parts': [{'text': 'This document is an INVOICE (confidence 0.97).'}]}
    ],
    'tools': [{
        'functionDeclarations': [
            {
                'name': 'classify_document',
                'description': 'Classify a document into a category',
                'parameters': {
                    'type': 'OBJECT',
                    'properties': {
                        'document_id': {'type': 'STRING'},
                        'confidence_threshold': {'type': 'NUMBER'}
                    },
                    'required': ['document_id']
                }
            }
        ]
    }]
}

with open('train_fc.jsonl', 'w') as f:
    f.write(json.dumps(fc_example))

print(json.dumps(fc_example, indent=2)[:500] + '...')
print('\nWrote train_fc.jsonl')

## Exercise 6: Pointwise evaluation

**Difficulty:** Medium

Use EvalTask with exact_match and rouge_l_sum. Compare base vs tuned on 20 examples.

1. Build a holdout eval dataset of prompt/reference pairs.
2. Run `EvalTask` with `exact_match` and `rouge_l_sum` on the base model.
3. Run the same task on the tuned endpoint.
4. Compare `summary_metrics` to see the tuned lift.

**Expected behaviour:** Two evaluation runs with summary_metrics showing tuned improvement.

In [ ]:
# Pointwise evaluation with vertexai.evaluation
import pandas as pd

def build_eval_dataset():
    '''Build evaluation dataset from holdout examples.'''
    return pd.DataFrame({
        'prompt': [
            'Classify: Invoice #2024-001, Amount: $4,590, Due: 2024-04-14',
            'Classify: Service Agreement between A and B, Term: 24 months',
            'Classify: Q3 Financial Report showing 18% YoY growth',
        ],
        'reference': ['INVOICE', 'CONTRACT', 'REPORT'],
    })

eval_df = build_eval_dataset()
print('Evaluation dataset:')
print(eval_df)

# Pointwise A/B pattern — runs once you have a tuned endpoint from Exercise 4.
def run_pointwise(eval_df, base_model='gemini-3.6-flash', tuned_endpoint=None):
    from vertexai.evaluation import EvalTask

    base_result = EvalTask(
        dataset=eval_df,
        metrics=['exact_match', 'rouge_l_sum'],
        experiment='ab-test',
    ).evaluate(model=base_model, experiment_run_name='base')

    tuned_result = EvalTask(
        dataset=eval_df,
        metrics=['exact_match', 'rouge_l_sum'],
        experiment='ab-test',
    ).evaluate(model=tuned_endpoint, experiment_run_name='tuned')

    print(f"Base  exact_match: {base_result.summary_metrics['exact_match']:.2%}")
    print(f"Tuned exact_match: {tuned_result.summary_metrics['exact_match']:.2%}")
    return base_result, tuned_result

print('\nrun_pointwise defined. After tuning:')
print('# base_result, tuned_result = run_pointwise(eval_df, tuned_endpoint=tuned_endpoint)')

## Exercise 7: Pairwise A/B test

**Difficulty:** Challenge

Run PairwiseMetric with baseline_model=base. Compute win_rate on 100 examples.

1. Define a `PairwiseMetric` (e.g. pairwise quality) with the base model as `baseline_model`.
2. Point the `EvalTask` candidate at the tuned endpoint.
3. Evaluate over the holdout set.
4. Read `candidate_model_win_rate` and `baseline_model_win_rate`.

**Expected behaviour:** candidate_model_win_rate and baseline_model_win_rate metrics.

In [ ]:
# Pairwise A/B test: tuned candidate vs base baseline
def run_pairwise(eval_df, base_model='gemini-3.6-flash', tuned_endpoint=None):
    from vertexai.evaluation import (
        EvalTask, PairwiseMetric, MetricPromptTemplateExamples,
    )

    pairwise_quality = PairwiseMetric(
        metric='pairwise_quality',
        metric_prompt_template=MetricPromptTemplateExamples.Pairwise.QUALITY,
        baseline_model=base_model,   # tuned candidate is judged against the base
    )

    result = EvalTask(
        dataset=eval_df,
        metrics=[pairwise_quality],
        experiment='ab-pairwise',
    ).evaluate(model=tuned_endpoint, experiment_run_name='tuned-vs-base')

    sm = result.summary_metrics
    print(f"Candidate (tuned) win rate: {sm.get('pairwise_quality/candidate_model_win_rate')}")
    print(f"Baseline (base)  win rate: {sm.get('pairwise_quality/baseline_model_win_rate')}")
    return result

print('run_pairwise defined. After tuning:')
print('# result = run_pairwise(eval_df, tuned_endpoint=tuned_endpoint)')
print('# (Use a 100-row holdout dataframe in place of eval_df for a real read.)')

## Exercise 8: Full DocuMind pipeline

**Difficulty:** Challenge

Generate data with Gemini 3.1 Pro -> validate JSONL -> fine-tune Flash -> A/B test -> decide deploy.

1. Use `gemini-3.1-pro-preview` to synthesize labelled examples.
2. Validate the JSONL, then launch a Flash tuning job.
3. Run the pointwise + pairwise A/B evaluation.
4. Apply the decision framework and print a deploy/no-deploy call.

**Expected behaviour:** End-to-end: 200 examples, tuned endpoint, win_rate > baseline, deployment decision.

In [ ]:
# Decision helper: should you fine-tune / deploy the tuned model?
def should_finetune(num_examples, monthly_calls, format_consistency_needed,
                    domain_specific_terms, requirements_stable):
    '''Returns recommendation + reasoning.'''
    score = 0
    reasons = []

    if num_examples >= 100:
        score += 2
        reasons.append(f'+ {num_examples} examples (>=100 threshold)')
    else:
        reasons.append(f'- Only {num_examples} examples (<100). Use few-shot instead')

    if monthly_calls > 50000:
        score += 2
        reasons.append(f'+ High volume ({monthly_calls:,}/month). Few-shot overhead expensive')

    if format_consistency_needed:
        score += 2
        reasons.append('+ Format consistency needed (JSON schemas, labels)')

    if domain_specific_terms:
        score += 1
        reasons.append('+ Domain-specific terminology')

    if not requirements_stable:
        score -= 3
        reasons.append('- Requirements change frequently. Prompts are faster to update')

    recommendation = 'FINE-TUNE' if score >= 3 else 'USE PROMPTING'
    return recommendation, reasons, score


# Step 1: synthesize labelled data with the high-reasoning Pro model
def generate_examples(n=5):
    prompt = ('Generate ' + str(n) + ' short synthetic business documents, one per line, '
              'each on its own line as "CATEGORY :: text", where CATEGORY is one of '
              'INVOICE, CONTRACT, REPORT, RECEIPT, CORRESPONDENCE.')
    resp = gen_client.models.generate_content(
        model='gemini-3.1-pro-preview',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.9,
            thinking_config=types.ThinkingConfig(thinking_budget=512),
        ),
    )
    return resp.text


# Step 2-5: orchestrate the end-to-end pipeline (guarded on real GCP resources)
def run_pipeline():
    print('1. Generating synthetic examples with gemini-3.1-pro-preview...')
    # synthetic = generate_examples(n=200)   # uncomment to actually call the model

    print('2. Validating train.jsonl...')
    errs = validate_jsonl('train.jsonl')
    print('   valid' if not errs else f'   {len(errs)} error(s)')

    print('3. Launching Flash tuning job... (uncomment to run)')
    # job = poll_until_complete(launch_tuning_job(train_uri=f'gs://{BUCKET}/train.jsonl'))
    # tuned_endpoint = job.tuned_model.endpoint

    print('4. A/B test tuned vs base... (uncomment to run)')
    # run_pointwise(build_eval_dataset(), tuned_endpoint=tuned_endpoint)
    # run_pairwise(build_eval_dataset(), tuned_endpoint=tuned_endpoint)

    print('5. Deployment decision:')
    rec, reasons, score = should_finetune(
        num_examples=200, monthly_calls=100000,
        format_consistency_needed=True, domain_specific_terms=True,
        requirements_stable=True)
    print(f'   Recommendation: {rec} (score {score})')
    for r in reasons:
        print(f'     {r}')
    print('   Deploy the tuned endpoint only if candidate_model_win_rate > baseline.')

run_pipeline()